# Storing Chat History in 3rd Party Storage

### By default, when using `ChatAgent`, chat history is stored in memory in the `AgentThread` object or the underlying inference service, if the service supports it
### But we need persistence chat history, for which the external database is to be used.
### Here we will use Redis database to store the chat history, the in-memory behavior will not be possible if you want to buid any real production agents

## First Step is to install Redis

In [3]:
# ensure you have docker installed
# Run Redis official image using docker
!docker run -d --name my-redis -p 6379:6379 redis:latest

dc0f046bef5dcbe90a9c70f22028b4e5d5ed3ddb210a88b3b060aa0a2b067d39


In [4]:
# check if the container is Running
!docker ps
# also you I can connect to the container to test it
# docker exec -it my-redis redis-cli

CONTAINER ID   IMAGE          COMMAND                  CREATED          STATUS          PORTS                                         NAMES
dc0f046bef5d   redis:latest   "docker-entrypoint.s…"   52 seconds ago   Up 51 seconds   0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp   my-redis


### Message storage and retrieval methods
- `add_message` - called to add new messages to the store
- `list_message` - called to retreive the messages from the store

`list_messages` should return the messages in ascending chronological order. All messages returned by it will be used by the ChatAgent when making calls to the underlying chat client. It's therefore important that this method considers the limits of the underlying model, and only returns as many messages as can be handled by the model.

Any chat history reduction logic, such as summarization or trimming, should be done before returning messages from `list_messages`.

### Serialization
You will have to create `ChatMessageStore` instance, this is the module which will be used to connect and attach to `AgentThread`, for the attach and resume of the thread.

To allow persisting threads, you need to implement two main methods ofthe `ChatMessageStore` protocol 
- `serialize_state`
- `deserialize-state`
These methods allow the store's state to be persisted and restored when resuming the thread

In [8]:
# This is the module which creates the RedisChatMessageStore
# The instance of this module is used to connect to the chat database and store the messages externally
from collections.abc import Sequence
from typing import Any
from uuid import uuid4
from pydantic import BaseModel
import json
import redis.asyncio as redis
from agent_framework import ChatMessage


class RedisStoreState(BaseModel):
    """State model for serializing and deserializing Redis chat message store data."""

    thread_id: str
    redis_url: str | None = None
    key_prefix: str = "chat_messages"
    max_messages: int | None = None


class RedisChatMessageStore:
    """Redis-backed implementation of ChatMessageStore using Redis Lists."""

    def __init__(
        self,
        redis_url: str | None = None,
        thread_id: str | None = None,
        key_prefix: str = "chat_messages",
        max_messages: int | None = None,
    ) -> None:
        """Initialize the Redis chat message store.

        Args:
            redis_url: Redis connection URL (for example, "redis://localhost:6379").
            thread_id: Unique identifier for this conversation thread.
                      If not provided, a UUID will be auto-generated.
            key_prefix: Prefix for Redis keys to namespace different applications.
            max_messages: Maximum number of messages to retain in Redis.
                         When exceeded, oldest messages are automatically trimmed.
        """
        if redis_url is None:
            raise ValueError("redis_url is required for Redis connection")

        self.redis_url = redis_url
        self.thread_id = thread_id or f"thread_{uuid4()}"
        self.key_prefix = key_prefix
        self.max_messages = max_messages

        # Initialize Redis client
        self._redis_client = redis.from_url(redis_url, decode_responses=True)

    @property
    def redis_key(self) -> str:
        """Get the Redis key for this thread's messages."""
        return f"{self.key_prefix}:{self.thread_id}"

    async def add_messages(self, messages: Sequence[ChatMessage]) -> None:
        """Add messages to the Redis store.

        Args:
            messages: Sequence of ChatMessage objects to add to the store.
        """
        if not messages:
            return

        # Serialize messages and add to Redis list
        serialized_messages = [self._serialize_message(msg) for msg in messages]
        await self._redis_client.rpush(self.redis_key, *serialized_messages)

        # Apply message limit if configured
        if self.max_messages is not None:
            current_count = await self._redis_client.llen(self.redis_key)
            if current_count > self.max_messages:
                # Keep only the most recent max_messages using LTRIM
                await self._redis_client.ltrim(self.redis_key, -self.max_messages, -1)

    async def list_messages(self) -> list[ChatMessage]:
        """Get all messages from the store in chronological order.

        Returns:
            List of ChatMessage objects in chronological order (oldest first).
        """
        # Retrieve all messages from Redis list (oldest to newest)
        redis_messages = await self._redis_client.lrange(self.redis_key, 0, -1)

        messages = []
        for serialized_message in redis_messages:
            message = self._deserialize_message(serialized_message)
            messages.append(message)

        return messages

    async def serialize_state(self, **kwargs: Any) -> Any:
        """Serialize the current store state for persistence.

        Returns:
            Dictionary containing serialized store configuration.
        """
        state = RedisStoreState(
            thread_id=self.thread_id,
            redis_url=self.redis_url,
            key_prefix=self.key_prefix,
            max_messages=self.max_messages,
        )
        return state.model_dump(**kwargs)

    async def deserialize_state(self, serialized_store_state: Any, **kwargs: Any) -> None:
        """Deserialize state data into this store instance.

        Args:
            serialized_store_state: Previously serialized state data.
            **kwargs: Additional arguments for deserialization.
        """
        if serialized_store_state:
            state = RedisStoreState.model_validate(serialized_store_state, **kwargs)
            self.thread_id = state.thread_id
            self.key_prefix = state.key_prefix
            self.max_messages = state.max_messages

            # Recreate Redis client if the URL changed
            if state.redis_url and state.redis_url != self.redis_url:
                self.redis_url = state.redis_url
                self._redis_client = redis.from_url(self.redis_url, decode_responses=True)

    def _serialize_message(self, message: ChatMessage) -> str:
        """Serialize a ChatMessage to JSON string."""
        message_dict = message.to_dict()
        return json.dumps(message_dict, separators=(",", ":"))

    def _deserialize_message(self, serialized_message: str) -> ChatMessage:
        """Deserialize a JSON string to ChatMessage."""
        message_dict = json.loads(serialized_message)
        return ChatMessage.from_dict(message_dict)

    async def clear(self) -> None:
        """Remove all messages from the store."""
        await self._redis_client.delete(self.redis_key)

    async def aclose(self) -> None:
        """Close the Redis connection."""
        await self._redis_client.aclose()

## Import Dependencies

Import the required libraries and Microsoft Agent Framework components:

- `asyncio`: For async/await support
- `os`: For accessing environment variables
- `json`: For JSON parsing
- `dotenv`: For loading environment variables from `.env` file
- `ChatAgent`: The main Agent class for building conversational AI agents
- `OpenAIChatClient`: Client for LLM inference using OpenAI-compatible endpoints (OpenRouter in this case)

In [9]:
# Import core dependencies to create the agent, for Agent Framework
import asyncio
import os
import json

from dotenv import load_dotenv, find_dotenv
# Core components for building Agent, tool-enabled agents
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient

## Load Environment Variables

Load environment variables from a `.env` file in the project directory. This file should contain:
- `OPENROUTER_ENDPOINT`: The OpenRouter API endpoint URL
- `OPENROUTER_API_KEY`: Your OpenRouter API key

In [10]:
# load environment file
load_dotenv(find_dotenv())

True

## Setup Chat Client

Configure the `OpenAIChatClient` to use OpenRouter API, which provides access to various LLM models including NVIDIA's Nemotron model. The client is configured with:

- `base_url`: The OpenRouter API endpoint
- `api_key`: Your API key for authentication
- `model_id`: The specific model to use (NVIDIA Nemotron 3 Nano 30B in this case)

In [11]:
# Setup OpenAIChatClient for LLM Inference - Here we will use OpenRouter API which is compatible with OpenAI and NVIDIA 30B model
# This client connects to the OpenRouter Models which are OpenAI-compatible endpoint
# Environment variables required
# OPENROUTER_ENDPOINT - 
# OPENROUTER_API_KEY
openai_chat_client = OpenAIChatClient(
    base_url=os.environ.get("OPENROUTER_ENDPOINT"),
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model_id="nvidia/nemotron-3-nano-30b-a3b:free"
)

In [12]:
AGENT_NAME = "FoodAgent"

AGENT_INSTRUCTIONS = """You are an expert AI Chef dedicated to helping users discover and prepare delicious meals. Keep it concise, short and effective.
"""

## Create the Food Agent

Create the first agent (`food_agent`) with:

- `name`: "FoodAgent"
- `chat_client`: The OpenAI chat client configured earlier
- `instructions`: The behavior instructions defined above
- `chat_message_store`: External Database Store for persistent message.

In [13]:
# Here we create the foodAgent
# create the agent remember we are not using any tools here, this is simple example
food_agent = ChatAgent(
    name = AGENT_NAME,
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
    chat_message_store_factory=lambda: RedisChatMessageStore(
        redis_url="redis://localhost:6379"
    )
)

In [14]:
#  Use the agent with persistent chat history
thread = food_agent.get_new_thread()
response = await food_agent.run("How to make Masala Chai, this is for Urula, Urula loves it", thread=thread)
print(response.text)

**Masala Chai for Urula**

| **Ingredient** | **Qty** |
|----------------|--------|
| Water          | 1 cup (240 ml) |
| Milk (full‑fat) | ½ cup (120 ml) |
| Strong black tea (e.g., Assam) | 1 tsp (or 1 bag) |
| Sugar | 1–2 tsp, to taste |
| Ground spices (adjust to taste) | ¼ tsp each: |
| • Cardamom | |
| • Cinnamon stick | |
| • Cloves | |
| • Fresh ginger (grated) | ¼ tsp or a small pinch |
| • Black pepper (optional) | pinch |

---

### Steps

1. **Boil spices & ginger**  
   - In a small saucepan, add ½ cup water, the spices, and grated ginger. Bring to a gentle boil for 1–2 min.

2. **Add tea**  
   - Stir in the tea leaves (or bag) and simmer another minute to steep.

3. **Add milk & sweetener**  
   - Pour in the milk and sugar. Return to a low boil, stirring occasionally.

4. **Final boil**  
   - Let it come to a rolling boil once more (about 30 seconds). This creates the frothy “kashmiri” top.

5. **Strain & serve**  
   - Pour through a fine sieve into mugs. Serve hot.

-

In [15]:
response2 = await food_agent.run("What does Urula loves?", thread=thread)

TypeError: TextReasoningContent.__init__() missing 1 required positional argument: 'text'